# Step 16.3: group cross-species correlations
CPU runtime. Upload the current `cross_species_rsa.py` to My Drive/rsa_colab. Existing step-16.2 HDF5 files stay in results_step16_2; original maps and configs are unnecessary. Each matrix entry is averaged equally across selected source participants. NaNs propagate. The output is a voxel-pair matrix, not a significance map.


In [ ]:
%pip -q install numpy nibabel PyYAML h5py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/rsa_colab')
sys.path.insert(0, str(ROOT))
from cross_species_rsa import group_cross_similarity, CorrelationReader


In [ ]:
# Source = rows, target = columns. Swap species and their settings to reverse.
SPECIE, TARGET_SPECIE = 'D', 'H'
RADIUS, TARGET_RADIUS = 4, 4
PARTICIPANTS = None  # all matching files; use [1, 2, ...] to require a cohort
TARGET_PARTICIPANTS = None  # optional validation against saved target cohort
MASK_TYPE = TARGET_MASK_TYPE = 'b_GreyMatter2mmB'
output = group_cross_similarity(
    ROOT / 'results_step16_2', dataset='EmoC', model='basic-block',
    rsa_class='all-mahalanobis', specie=SPECIE, target_specie=TARGET_SPECIE,
    radius=RADIUS, target_radius=TARGET_RADIUS, mask_type=MASK_TYPE,
    target_mask_type=TARGET_MASK_TYPE, participants=PARTICIPANTS,
    target_participants=TARGET_PARTICIPANTS,
    source_batch=128, target_batch=4096, verbose=True)
print(output)


The result is saved beside the participant folders under `<source>-group/`, with the same HDF5 filename. Every selected input must be complete and use identical voxel ordering, pairs, target participants and target RDMs. An explicit cohort fails if any file is missing. Re-running recomputes and atomically replaces the group result.

Optional: extract a target-space group map for one source voxel with `CorrelationReader(output).target_map(...)` using a context manager.


## Step 16.4: source-space summary maps
Run after step 16.3. Twenty thresholds include the finite minimum and maximum. Counts use strict >; means exclude undefined correlations. Saves 21 NIfTI maps plus threshold metadata to the class folder under mean/.


In [ ]:
from cross_species_rsa import summarize_group_cross_similarity
maps = summarize_group_cross_similarity(output, verbose=True)
print(maps[0].parent)
